# Per-model GPU utilisation profiling

Profiles a single forward pass of every available model architecture with
PyTorch's `torch.profiler`, printing a CPU/CUDA time and memory breakdown
table per model. This is an operator-level view (where does time go
*inside* a model), complementary to the wall-clock throughput/memory
benchmarking in `src/benchmark.py` (used by `sacair_2026`).

In [1]:
import gc

import torch
from torch.profiler import ProfilerActivity, profile, record_function

from src.models import avail_models, get_model
from src.preprocess import Instance
from src.run_types import CUTOFF_9_NAMES, CentreCropConfig, OG_Sampler
from src.stats import get_all_sets
from src.utils import load_rgb_frames_from_video
from src.video_dataset import get_transform, get_video_path, get_wlasl_info

## Setup

Frame count is model-dependent, not a fixed choice: most models use 32
frames iff `32x3` is in their name, else 16 -- except the `_e`/`_r`
(interpolated positional encoding, extended/reduced) variants, which flip
that. See `sacair_2026/benchmark.ipynb`'s model list for the same
convention.

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

split_name = CUTOFF_9_NAMES[0] # asl100_cutoff_9
frame_size = 224
batch_size = 2
num_classes = 100 # asl100_cutoff_9
row_limit = 20 # top operators by time, per model

model_names = avail_models()
print(', '.join(model_names))

cuda
S3D, R3D_18, R(2+1)D_18, Swin3D_T, Swin3D_S, Swin3D_B, MViTv2_S, MViTv2_S_e, MViTv1_B, MViTv2_S_16x4, MViTv2_S_16x4_e, MViTv2_B_32x3, MViTv2_B_32x3_r


In [3]:
def frame_count_for(arch: str) -> int:
    """Most models take 32 frames iff '32x3' is in their name, else 16 --
    except _e/_r (interpolated positional encoding) suffixes, which flip that.
    """
    base_32 = '32x3' in arch
    flipped = arch.endswith(('_e', '_r'))
    return (16 if base_32 else 32) if flipped else (32 if base_32 else 16)


all_sets = get_all_sets(split_name)
wlasl_info = get_wlasl_info(split_name, 'test')
sample_instance = Instance.model_validate(all_sets['test'][0]['instances'][0])
sample_path = get_video_path(sample_instance.video_id, wlasl_info['root'])
raw_frames = load_rgb_frames_from_video(
    sample_path, sample_instance.frame_start, sample_instance.frame_end
)

def make_batch(num_frames: int) -> torch.Tensor:
    """A real, correctly-normalised, correctly-shaped batch of clips."""
    transform, _, _ = get_transform(
        temporal_aug=[OG_Sampler(target_length=num_frames)],
        spatial_aug=[CentreCropConfig(frame_size=frame_size)],
        normalise_to_float=True,
        permute_time_channel=True,
    )
    clip = transform(raw_frames)
    return clip.unsqueeze(0).repeat(batch_size, *([1] * clip.dim())).to(device)

batches = {16: make_batch(16), 32: make_batch(32)}
for num_frames, batch in batches.items():
    print(f'{num_frames}-frame batch shape: {tuple(batch.shape)}')

16-frame batch shape: (2, 3, 16, 224, 224)
32-frame batch shape: (2, 3, 32, 224, 224)


## Profile each model

In [4]:
def profile_model(model: torch.nn.Module, inputs: torch.Tensor, title: str) -> None:
    """Profile one forward pass, printing a CPU/CUDA time and memory breakdown table."""
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)
    sort_by = 'cuda_time_total' if torch.cuda.is_available() else 'cpu_time_total'

    model = model.to(device).eval()
    with (
        profile(activities=activities, record_shapes=True, profile_memory=True) as prof,
        record_function(f'{title}_inference'),
        torch.no_grad(),
    ):
        model(inputs)

    print(prof.key_averages().table(sort_by=sort_by, row_limit=row_limit))

In [5]:
for arch in model_names:
    num_frames = frame_count_for(arch)
    print('='*80)
    print(arch, f'({num_frames} frames)')
    print('='*80)
    model = None
    try:
        model = get_model(arch, num_classes, drop_p=0.0)
        profile_model(model, batches[num_frames], arch)
    except Exception as e:  # noqa: BLE001 -- sweeping many independent model
        # implementations; one shape/API mismatch shouldn't stop the rest.
        print(f'Skipped {arch}: {e}')
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    print()

S3D (16 frames)


[W914 21:02:35.787261196 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)
[W914 21:02:35.787275865 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)


---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    S3D_inference         2.55%       4.695ms       100.00%     184.256ms     184.256ms       2.429ms         1.32%     184.259ms     184.259ms           0 b           0 b           0 b    -824.34 Mb             1  
                     aten::conv3d         0.18%     338.660us        8

[W914 21:02:36.372275089 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)
[W914 21:02:36.372290440 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)


--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                R3D_18_inference         1.62%       1.155ms       100.00%      71.340ms      71.340ms     283.000us         0.40%      71.343ms      71.343ms           0 b           0 b       8.12 Mb      -1.27 Gb             1  
                    aten::conv3d         0.12%      88.425us        92.22%

...

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
        MViTv2_S_16x4_e_inference         4.63%      13.477ms       100.00%     291.289ms     291.289ms       4.698ms         1.59%     294.645ms     294.645ms           0 b    -117.79 Kb           0 b     -18.13 Gb             1  
                      aten::copy_        85.64%     249.463ms        8

[W914 21:02:46.339506018 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)
[W914 21:02:46.339521170 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)


---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
          MViTv2_B_32x3_inference         4.86%      20.001ms       100.00%     411.794ms     411.794ms       6.975ms         1.68%     415.139ms     415.139ms           0 b    -176.53 Kb           0 b     -25.45 Gb             1  
                      aten::copy_        83.77%     344.959ms        8

[W914 21:02:47.246389866 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)
[W914 21:02:47.246404173 kineto_shim.cpp:415] Adding profiling metadata requires using torch.profiler with Kineto support (USE_KINETO=1)


---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
        MViTv2_B_32x3_r_inference        11.41%      20.517ms       100.00%     179.866ms     179.866ms       6.917ms         3.82%     181.242ms     181.242ms           0 b    -118.07 Kb           0 b      -8.93 Gb             1  
                     aten::linear         0.66%       1.180ms         